**INSTALL LIBRARIES**

In [1]:
!pip install scikit-learn pandas numpy matplotlib seaborn gradio joblib

**IMPORT LIBRARIES**

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

import gradio as gr

**LOAD DATASET**

Dataset:
Telco Customer Churn Dataset

Download from:
IBM Telco Customer Churn Dataset

Upload Dataset to Colab

In [5]:
!unzip archive.zip
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

Archive:  archive.zip
  inflating: WA_Fn-UseC_-Telco-Customer-Churn.csv  


In [6]:
df = pd.read_csv(
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

**UNDERSTAND DATASET**

In [7]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Check columns

In [8]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

Check missing values

In [9]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


**DATA CLEANING**

Step 1 — Remove Customer ID

In [10]:
df.drop("customerID", axis=1, inplace=True)

Step 2 — Convert TotalCharges

In [11]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

Step 3 — Convert Target Variable

In [12]:
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

**SPLIT FEATURES AND TARGET**

In [13]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

**TRAIN TEST SPLIT**

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

**IDENTIFY COLUMN TYPES**

In [15]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns

numerical_cols = X.select_dtypes(
    exclude=["object"]
).columns

**CREATE PREPROCESSING PIPELINES**

Numerical Pipeline

In [16]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

Categorial Pipeline

In [17]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

Combine Pipelines

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_cols
        ),
        (
            "cat",
            categorical_transformer,
            categorical_cols
        )
    ]
)

**CREATE MODEL PIPELINES**

Logistic Regression Pipeline

In [19]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter=1000)
        )
    ]
)

Random Forest Pipeline

In [20]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier()
        )
    ]
)

**TRAIN MODELS**

In [21]:
logistic_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('classifier', RandomForestClassifier())])

**EVALUATE MODELS**

Logistic Regression Evaluation

In [22]:
logistic_preds = logistic_pipeline.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, logistic_preds)
)

print(
    classification_report(
        y_test,
        logistic_preds
    )
)

Accuracy: 0.8211497515968772
              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.69      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.82      0.82      0.82      1409



Random Forest Evaluation

In [23]:
rf_preds = rf_pipeline.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, rf_preds)
)

print(
    classification_report(
        y_test,
        rf_preds
    )
)

Accuracy: 0.7913413768630234
              precision    recall  f1-score   support

           0       0.82      0.91      0.87      1036
           1       0.65      0.46      0.54       373

    accuracy                           0.79      1409
   macro avg       0.74      0.69      0.70      1409
weighted avg       0.78      0.79      0.78      1409



**HYPERPARAMETER TUNING**

Use:

GridSearchCV

Logistic Regression Grid Search

In [24]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(
    logistic_pipeline,
    param_grid,
    cv=3,
    scoring="accuracy"
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('o...
                                                                         Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                                       ('classifier',
                                        LogisticRegression(max_iter=1000))]),
             param_grid={'classifier__C': [0.01, 0.1, 1, 10]},
             scoring='accuracy')

Best Parameters

In [25]:
print(grid_search.best_params_)

{'classifier__C': 10}


Best Score

In [26]:
print(grid_search.best_score_)

0.8020944266950657


Save best model:

In [27]:
joblib.dump(
    grid_search.best_estimator_,
    "churn_pipeline.joblib"
)

['churn_pipeline.joblib']

It contains:

- preprocessing
- scaling
- encoding
- trained model

all in ONE reusable pipeline.

**TEST SAVED PIPELINE**

Load Pipeline

In [28]:
model = joblib.load(
    "churn_pipeline.joblib"
)

Predict:

In [29]:
sample = X_test.iloc[[0]]

prediction = model.predict(sample)

print(prediction)

[1]


In [32]:
print(X.columns)

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='object')


**CREATE app.py**

Create:

In [30]:
%%writefile app.py

Writing app.py


In [33]:
import gradio as gr
import pandas as pd
import joblib

model = joblib.load("churn_pipeline.joblib")

# Get expected columns from trained model
expected_columns = model.feature_names_in_

def predict_churn(
    gender,
    seniorcitizen,
    tenure,
    monthlycharges,
    totalcharges,
    contract
):

    # Create empty row with ALL required columns
    input_dict = {col: 0 for col in expected_columns}

    # Fill only known fields from UI
    input_dict["gender"] = gender
    input_dict["SeniorCitizen"] = seniorcitizen
    input_dict["tenure"] = tenure
    input_dict["MonthlyCharges"] = monthlycharges
    input_dict["TotalCharges"] = totalcharges
    input_dict["Contract"] = contract

    # Convert to DataFrame
    input_df = pd.DataFrame([input_dict])

    # Predict
    prediction = model.predict(input_df)[0]

    return (
        "Customer is likely to churn."
        if prediction == 1
        else "Customer is not likely to churn."
    )

interface = gr.Interface(
    fn=predict_churn,
    inputs=[
        gr.Dropdown(["Male", "Female"], label="Gender"),
        gr.Number(label="Senior Citizen (0 or 1)"),
        gr.Number(label="Tenure"),
        gr.Number(label="Monthly Charges"),
        gr.Number(label="Total Charges"),
        gr.Dropdown(["Month-to-month", "One year", "Two year"], label="Contract")
    ],
    outputs="text",
    title="Customer Churn Prediction"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f6ca1f0cd5f7db403b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
